# 05 — Inference and prediction saving

**Purpose:** Load a saved LoRA adapter or a fully tuned model, generate one category per example, and save raw outputs and metadata.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# Kaggle/Colab project setup — NO PIP INSTALLATIONS.
# Uses the versions already installed in this runtime.
from pathlib import Path
from importlib import metadata
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PULL_LATEST = False  # Enable only for a clean clone after pushing fixes to GitHub.
if Path("/kaggle/working").is_dir():
    WORKSPACE = Path("/kaggle/working")
elif Path("/content").is_dir():
    WORKSPACE = Path("/content")
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / "nlp_research_proj"
ROOT = REPO_DIR / "Urdu_Qwen_Research_Notebooks"

if (REPO_DIR / ".git").is_dir():
    if PULL_LATEST:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"Existing non-git folder: {REPO_DIR}; inspect before removing it")
elif (Path.cwd() / "project_utils.py").is_file():
    ROOT = Path.cwd()
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(f"Project code missing from {ROOT}")

# Check user-confirmed Kaggle versions; NEVER uninstall or upgrade preinstalled dependencies.
EXPECTED = {
    "transformers": "5.16.1",
    "peft": "0.20.0",
    "torch": "2.11.0",  # the Kaggle build has a +cu128 suffix
    "numpy": "2.1.3",
    "scipy": "1.16.3",
    "scikit-learn": "1.6.1",
}
versions = {}
for name, expected in EXPECTED.items():
    try:
        got = metadata.version(name)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Missing preinstalled dependency: {name}") from None
    versions[name] = got
    if got.split("+")[0] != expected:
        raise RuntimeError(
            f"This code is checked for {name}={expected}, found {got}. "
            "Do NOT run pip; use the tested Kaggle image or review the API changes."
        )
for name in ("accelerate", "datasets", "pandas", "matplotlib", "pytest"):
    try:
        versions[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        raise RuntimeError(f"Required package missing: {name}. This runtime cannot run the selected notebook.") from None
print("Using preinstalled Kaggle libraries (no installations):", versions)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
api = check_training_api()
print("Project ready:", ROOT)
print("Training API ready:", api["transformers"], api["trainer_tokenizer_key"], api["warmup_key"])


Run once per trained checkpoint and language. **Only change `SPLIT` to `test` after finalizing training settings.** Run the English test on the same checkpoint if examining retention. The baseline model may use this notebook with `METHOD="base"`.

In [ ]:
import torch
TASK='sib200'; LANG='ur'; SPLIT='validation'  # finally: 'test'
METHOD='lora'  # 'lora', 'full', or 'base'
RUN_ID='sib200_lora_ur_n10_s42_r8'
MAX_ROWS=None  # 10 for smoke, None for full split
labels = task_labels(TASK)
rows = load_task(TASK,LANG,SPLIT)
if MAX_ROWS is not None: rows=rows[:MAX_ROWS]
model_id=cfg()['model_id']; checkpoint=None
if METHOD!='base':
    state=read_json(ROOT/'outputs/checkpoints'/RUN_ID/'complete.json')
    if state['method'] != METHOD or state['task'] != TASK: raise ValueError('Checkpoint metadata does not match method/task')
    model_id=state['base_model']; checkpoint=Path(state['saved_path'])
    if not checkpoint.exists(): checkpoint=ROOT/'outputs/checkpoints'/RUN_ID/'model'
    if not checkpoint.exists(): raise FileNotFoundError(checkpoint)
    if METHOD=='full':
        model,tokenizer=load_model(str(checkpoint))
    else:
        model,tokenizer=load_model(model_id,adapter_path=str(checkpoint))
else:
    model,tokenizer=load_model(model_id)
print('Model:',model_id,'mode:',METHOD,'rows:',len(rows))

In [ ]:
path=ROOT/'outputs/predictions'/f'{RUN_ID}_{LANG}_{SPLIT}.csv'
meta=dict(task=TASK,lang=LANG,split=SPLIT,model_id=model_id,
          run_id=RUN_ID,method=METHOD,checkpoint=str(checkpoint) if checkpoint else None,
          dataset_sha256=dataset_sha(TASK,LANG,SPLIT),subset=(MAX_ROWS is not None),
          decoding={'do_sample':False,'max_new_tokens':cfg()['max_new_tokens']})
outputs=infer_resume(model,tokenizer,rows,labels,LANG,path,meta,cfg()['max_new_tokens'])
print('Saved:',path, 'count:',len(outputs),'invalid:',sum(not x['valid'] for x in outputs))
print('First outputs:',[(p['reference'],p['prediction'],p['raw_prediction']) for p in outputs[:8]])

In [ ]:
del model
if torch.cuda.is_available(): torch.cuda.empty_cache()